## Random Forest

O [Random Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) é um modelo de ensemble que combina múltiplas árvores de decisão para melhorar a performance e reduzir o overfitting.

Cada árvore é treinada em uma amostra diferente dos dados, e a decisão final é feita por votação.

Esse modelo tende a ser mais robusto e apresentar melhor generalização em comparação com uma única árvore.

### Hiperparâmetros utilizados

- **n_estimators**: número de árvores na floresta.
  - Mais árvores → melhor desempenho (até certo ponto)

- **max_depth**: profundidade máxima das árvores.
  - Controla o overfitting

- **min_samples_split**: mínimo de amostras para dividir um nó

- **min_samples_leaf**: O número mínimo de amostras necessário para que um nó seja uma folha

### Estratégia

Foi utilizado GridSearchCV para encontrar a melhor combinação de hiperparâmetros, utilizando validação cruzada e a métrica ROC AUC.

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

from preprocessing import load_and_preprocess

X_train, X_test, y_train, y_test = load_and_preprocess(
    "../predictive_models/scrdata_202505.csv"
)

param_grid_rf = {
    "n_estimators": [700, 800, 900],
    "max_depth": [50, 60, 70],
    "min_samples_split": [32, 34, 36],
    "min_samples_leaf": [12, 14, 16]
}

grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid_rf,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print("Melhores parâmetros:", grid_rf.best_params_)
print("Melhor score (CV):", grid_rf.best_score_)

best_rf = grid_rf.best_estimator_

y_pred = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:, 1]

roc = roc_auc_score(y_test, y_proba)
f1 = f1_score(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)

resultados = {
    "modelo": "RandomForest",
    "roc_auc": roc,
    "f1": f1,
    "accuracy": acc,
    "roc_auc_cv": grid_rf.best_score_,
    "best_params": str(grid_rf.best_params_)
}

df_result = pd.DataFrame([resultados])
df_result.to_csv("resultados_rf.csv", index=False, sep=";")

FileNotFoundError: [Errno 2] No such file or directory: 'predictive_models/scrdata_202505.csv'